In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catchments = spark.table("scottish_housing_ws.silver.school_catchments")
postcodes = spark.table("scottish_housing_ws.silver.postcode_lookup")
hpi = spark.table("scottish_housing_ws.silver.uk_hpi")

In [0]:
catchments.printSchema()
postcodes.printSchema()

In [0]:
catchment_postcodes = (
    catchments
    .join(
        postcodes.select(
            "postcode",
            "urban_rural_class",
            "simd_2020_rank",
            "data_zone_2022"
        ),
        on="postcode",
        how="left"
    )
)

In [0]:
# One row per school catchment (seed_code)
# Aggregate postcode-level attributes to catchment-level profile
# urban_rural_class: 1-2 = large urban, 3-4 = small town, 5-6 = rural
# simd_2020_rank: 1 = most deprived, highest = least deprived

#   - median SIMD rank (more robust than mean for deprivation data)
#   - pct postcodes in large urban areas (class 1 or 2)
#   - pct postcodes in most deprived quintile (bottom 20% = rank <= 1396)
#   - total postcode count in catchment

# Note: row count exceeds 328 schools because some catchments straddle council boundaries. Postcodes in the same catchment but different council areas produce separate rows. This is a genuine data characteristic.

catchment_profiles = (
    catchment_postcodes
    .groupBy(
        "seed_code",
        "secondary_school_name",
        "school_local_authority",
        "council_area_code"
    )
    .agg(
        F.count("postcode").alias("postcode_count"),
        F.percentile_approx("simd_2020_rank", 0.5).alias("median_simd_rank"),
        F.round(F.avg("simd_2020_rank"), 0).alias("mean_simd_rank"),
        F.round(
            F.sum(
                F.when(F.col("urban_rural_class").isin(1, 2), 1).otherwise(0)
            ) / F.count("postcode") * 100, 1
        ).alias("pct_large_urban"),
        F.round(
            F.sum(
                F.when(F.col("urban_rural_class").isin(5, 6, 7, 8), 1).otherwise(0)
            ) / F.count("postcode") * 100, 1
        ).alias("pct_rural"),
        F.round(
            F.sum(
                F.when(F.col("simd_2020_rank") <= 1396, 1).otherwise(0)
            ) / F.count("postcode") * 100, 1
        ).alias("pct_most_deprived_quintile"),
        F.round(
            F.sum(
                F.when(F.col("simd_2020_rank") >= 5584, 1).otherwise(0)
            ) / F.count("postcode") * 100, 1
        ).alias("pct_least_deprived_quintile")
    )
)

In [0]:
#Get most recent council-level HPI prices 
latest_month = (
    hpi
    .filter(F.col("area_code").startswith("S12"))
    .groupBy("area_code")
    .agg(F.max("year_month").alias("latest_year_month"))
)

hpi_latest = (
    hpi
    .join(latest_month, on="area_code", how="inner")
    .filter(F.col("year_month") == F.col("latest_year_month"))
    .select(
        F.col("area_code").alias("council_area_code"),
        F.col("year_month").alias("hpi_year_month"),
        F.col("average_price").alias("council_avg_price")
    )
)

In [0]:
gold = (
    catchment_profiles
    .join(hpi_latest, on="council_area_code", how="left")
    .select(
        "seed_code",
        "secondary_school_name",
        "school_local_authority",
        "council_area_code",
        "postcode_count",
        "median_simd_rank",
        "mean_simd_rank",
        "pct_large_urban",
        "pct_rural",
        "pct_most_deprived_quintile",
        "pct_least_deprived_quintile",
        "council_avg_price",
        "hpi_year_month"
    )
    .orderBy("school_local_authority", "secondary_school_name")
)

In [0]:
print(f"Row count: {gold.count()}")

# Should be ~328 rows, one per catchment
# Check Edinburgh schools as a sanity check
gold.filter(F.col("school_local_authority") == "City of Edinburgh").show(10, truncate=False)

In [0]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.gold.catchment_characterisation")
)

In [0]:
(
    gold.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("abfss://data@scottishhousingdl.dfs.core.windows.net/exports/gold_catchment_characterisation/")
)

print("Gold 11 complete.")

In [0]:
# Catchment Characterisation
# Sources: silver.school_catchments, silver.postcode_lookup, silver.uk_hpi
# Grain: one row per secondary school catchment (328 schools)

# Builds a profile of each school catchment area by aggregating postcode-level deprivation, urban/rural, and transport characteristics across all postcodes within the catchment.

# Joins to council-level HPI (most recent month) to give a price context for the council area each catchment sits in.

# Limitation: no property-level transaction data available (RoS Cloudflare protected). Cannot compute actual per-catchment price premiums.
# This table characterises catchments; it does not price them.